# Group 4 Challenge: Campus Digital Media and Loan Queue

Your team is building the inventory and overdue-fine engine for the university media library.
Fix the broken starter code across all three tasks so every assertion passes.

**Scoring:** Task 1 (`30 pts`) + Task 2 (`35 pts`) + Task 3 (`35 pts`) = **`100 pts`**.

In [ ]:
SCORES = {"task_1": 0, "task_2": 0, "task_3": 0}
print("Score tracker initialized. Complete Tasks 1 to 3 below!")

---
## Task 1: Encapsulation `[30 pts]` — Protect `MediaItem` State

Fix the `MediaItem` class below to enforce these rules:
1. `__init__(self, item_id: str, title: str, copies: int)`:
   - Raise `ValueError("copies cannot be negative")` if `copies < 0`.
   - Store `self._item_id`, `self._title`, `self._copies`, and `self._borrowers = []`.
2. Read-only properties:
   - `item_id` -> returns `self._item_id`
   - `borrowers` -> returns `tuple(self._borrowers)` (prevents callers from mutating internal list!)
3. Validated `copies` property:
   - `@property` `copies` returns `self._copies`.
   - `@copies.setter` raises `ValueError("copies cannot be negative")` if `value < 0`; else sets `self._copies`.
4. Method `checkout(self, student_name: str)`:
   - Raise `ValueError("no copies available")` if `self._copies == 0`.
   - Decrement `self._copies -= 1` and append `student_name` to `self._borrowers`.

In [ ]:
class MediaItem:
    def __init__(self, item_id: str, title: str, copies: int):
        # BUG 1: Missing negative copies validation
        # BUG 2: Public attributes instead of protected (_item_id, _title, _copies, _borrowers)
        self.item_id = item_id
        self.title = title
        self.copies = copies
        self.borrowers = []

    # TODO: Implement read-only properties: item_id, borrowers (returning a tuple)
    # TODO: Implement copies @property and @copies.setter with validation (>= 0)

    def checkout(self, student_name: str):
        # BUG 3: Missing availability check (copies == 0)
        self.copies -= 1
        self.borrowers.append(student_name)

    def late_fee(self, days_late: int) -> float:
        return 0.0

    def catalog_entry(self) -> str:
        return f"[{self._item_id}] {self._title}"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 1 (30 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    bad_init = False
    try:
        MediaItem("M-0", "Broken", -1)
    except ValueError:
        bad_init = True
    assert bad_init, "MediaItem with negative copies must raise ValueError."

    item = MediaItem("BK-10", "Python Design", 1)
    assert hasattr(item, "_item_id") and hasattr(item, "_copies") and hasattr(item, "_borrowers"), "Use protected attributes (_item_id, _copies, _borrowers)."
    assert item.item_id == "BK-10" and item.copies == 1, "Properties item_id and copies must return initial values."

    ro = False
    try:
        item.item_id = "NEW"
    except AttributeError:
        ro = True
    assert ro, "Property 'item_id' must be read-only."

    assert isinstance(item.borrowers, tuple), "Property 'borrowers' must return a tuple."

    bad_set = False
    try:
        item.copies = -3
    except ValueError:
        bad_set = True
    assert bad_set, "Setting copies < 0 must raise ValueError."

    item.checkout("Ana")
    assert item.copies == 0 and item.borrowers == ("Ana",), "checkout() must decrement copies and record borrower."

    empty_err = False
    try:
        item.checkout("Ben")
    except ValueError:
        empty_err = True
    assert empty_err, "checkout() on 0 copies must raise ValueError."

    SCORES["task_1"] = 30
    print("[PASS] Task 1 (30 / 30 pts): MediaItem encapsulation verified.")
except Exception as e:
    SCORES["task_1"] = 0
    print(f"[FAIL] Task 1 (0 / 30 pts): {e}")

---
## Task 2: Inheritance `[35 pts]` — `PrintedBook` and `AudioBook`

Fix `PrintedBook` and `AudioBook` so they inherit from `MediaItem`:
1. `PrintedBook(MediaItem)`:
   - `__init__(self, item_id: str, title: str, copies: int, shelf_code: str)`: call `super().__init__(item_id, title, copies)` and store `self.shelf_code = shelf_code`.
   - Override `late_fee(self, days_late: int) -> float`: return `round(max(0, days_late) * 5.0, 2)`.
   - Extend `catalog_entry(self) -> str`: return `f"{super().catalog_entry()} | Print (Shelf {self.shelf_code})"`.
2. `AudioBook(MediaItem)`:
   - `__init__(self, item_id: str, title: str, copies: int, duration_mins: int)`: call `super().__init__(item_id, title, copies)` and store `self.duration_mins = duration_mins`.
   - Override `late_fee(self, days_late: int) -> float`: return `round(max(0, days_late) * 2.5, 2)`.
   - Extend `catalog_entry(self) -> str`: return `f"{super().catalog_entry()} | Audio ({self.duration_mins}m)"`.

In [ ]:
class PrintedBook(MediaItem):
    def __init__(self, item_id: str, title: str, copies: int, shelf_code: str):
        # BUG 1: Missing super().__init__(item_id, title, copies)
        self.shelf_code = shelf_code

    def late_fee(self, days_late: int) -> float:
        # BUG 2: Return max(0, days_late) * 5.0
        return 0.0

    def catalog_entry(self) -> str:
        # BUG 3: Extend super().catalog_entry()
        return self.shelf_code


class AudioBook(MediaItem):
    def __init__(self, item_id: str, title: str, copies: int, duration_mins: int):
        # BUG 1: Missing super().__init__(item_id, title, copies)
        self.duration_mins = duration_mins

    def late_fee(self, days_late: int) -> float:
        # BUG 2: Return max(0, days_late) * 2.5
        return 0.0

    def catalog_entry(self) -> str:
        # BUG 3: Extend super().catalog_entry()
        return f"{self.duration_mins}m"

In [ ]:
# RUN THIS CELL TO VERIFY TASK 2 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert issubclass(PrintedBook, MediaItem) and issubclass(AudioBook, MediaItem), "Both subclasses must inherit from MediaItem."

    pb = PrintedBook("P-1", "Clean Code", 3, "CS-A4")
    ab = AudioBook("A-1", "Deep Work", 2, 320)

    assert pb.item_id == "P-1" and pb.late_fee(4) == 20.0, "PrintedBook late_fee(4) must be 20.0."
    assert ab.item_id == "A-1" and ab.late_fee(4) == 10.0, "AudioBook late_fee(4) must be 10.0."
    assert pb.late_fee(-2) == 0.0, "Negative days_late must return 0.0."

    assert pb.catalog_entry() == "[P-1] Clean Code | Print (Shelf CS-A4)", f"Unexpected catalog_entry: {pb.catalog_entry()}"
    assert ab.catalog_entry() == "[A-1] Deep Work | Audio (320m)", f"Unexpected catalog_entry: {ab.catalog_entry()}"

    SCORES["task_2"] = 35
    print("[PASS] Task 2 (35 / 35 pts): Inheritance and super() verified.")
except Exception as e:
    SCORES["task_2"] = 0
    print(f"[FAIL] Task 2 (0 / 35 pts): {e}")

---
## Task 3: Polymorphism and Duck Typing `[35 pts]` — `LabEquipmentKit` & `total_overdue_fines`

Fix the code below to achieve clean polymorphism:
1. `LabEquipmentKit` must **NOT** inherit from `MediaItem` (standalone duck-typed class).
   - Rename/implement `late_fee(self, days_late: int) -> float` returning `round(max(0, days_late) * self.daily_rate, 2)`.
2. Refactor `total_overdue_fines(items: list, days_late: int) -> float`:
   - Remove **all** `isinstance(...)` checks!
   - Loop through `items`, sum `item.late_fee(days_late)`, and return `round(total, 2)`.

In [ ]:
class LabEquipmentKit:
    def __init__(self, kit_name: str, daily_rate: float):
        self.kit_name = kit_name
        self.daily_rate = daily_rate

    # BUG 1: Wrong method name for duck typing! Should be late_fee(self, days_late: int) -> float
    def compute_penalty(self, days_late: int) -> float:
        return round(max(0, days_late) * self.daily_rate, 2)


def total_overdue_fines(items: list, days_late: int) -> float:
    # BUG 2: Fragile isinstance() branching misses LabEquipmentKit!
    # Refactor into a clean polymorphic loop with ZERO isinstance() calls.
    total = 0.0
    for item in items:
        if isinstance(item, PrintedBook):
            total += item.late_fee(days_late)
        elif isinstance(item, AudioBook):
            total += item.late_fee(days_late)
    return round(total, 2)

In [ ]:
# RUN THIS CELL TO VERIFY TASK 3 (35 pts)
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
try:
    assert not issubclass(LabEquipmentKit, MediaItem), "LabEquipmentKit must NOT inherit from MediaItem."
    assert hasattr(LabEquipmentKit, "late_fee"), "LabEquipmentKit must implement late_fee()."

    assert "isinstance" not in total_overdue_fines.__code__.co_names, "Remove all isinstance() calls from total_overdue_fines()."

    items = [
        PrintedBook("P2", "Stats", 1, "B1"),   # 3 * 5.0 = 15.0
        AudioBook("A2", "ML", 1, 180),        # 3 * 2.5 = 7.5
        LabEquipmentKit("Arduino Kit", 10.0), # 3 * 10.0 = 30.0
    ]
    fines = total_overdue_fines(items, 3)
    assert fines == 52.5, f"Expected total fines 52.5, got {fines}."

    SCORES["task_3"] = 35
    print("[PASS] Task 3 (35 / 35 pts): Polymorphism and duck typing verified.")
except Exception as e:
    SCORES["task_3"] = 0
    print(f"[FAIL] Task 3 (0 / 35 pts): {e}")

---
## Final Scoreboard

In [ ]:
# RUN THIS CELL TO VIEW YOUR FINAL COMPETITION SCORE
SCORES = globals().setdefault("SCORES", {"task_1": 0, "task_2": 0, "task_3": 0})
total = SCORES["task_1"] + SCORES["task_2"] + SCORES["task_3"]
print("=" * 42)
print("DS212 GROUP COMPETITION SCOREBOARD")
print("=" * 42)
print(f"Task 1 (Encapsulation): {SCORES['task_1']:>2} / 30 pts")
print(f"Task 2 (Inheritance):   {SCORES['task_2']:>2} / 35 pts")
print(f"Task 3 (Polymorphism):  {SCORES['task_3']:>2} / 35 pts")
print("-" * 42)
print(f"TOTAL SCORE:           {total:>3} / 100 pts")
if total == 100:
    print("STATUS: ALL CHALLENGES COMPLETED [100%]")
else:
    print("STATUS: IN PROGRESS — KEEP DEBUGGING!")
print("=" * 42)